In [1]:
'''
task: classify syllogism validity with NL + CLIF notation
models: gemma-2-2b-it
dataset: pfolio
evaluation: zero-shot
'''
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# start preparing for QA pipeline
! pip install -U accelerate
! pip install -U transformers
!pip install transformers
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 154.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.0 MB/s eta 0:00:00


In [3]:
import pandas as pd

pfolio_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/p-folio/data/pfolio_kr_gold_train.csv")

In [4]:
# evaluation metrics

import numpy as np
import re
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

def predict_answer(model, tokenizer, obj, subject, obj_clif, subject_clif, ref_relation=None, source_knowledge=None):
  # define notation grammar
  grammar = r"""
    start: program
    program: [stat]+
    stat: proposition newline | keyword* quantifier* symbol* leftparen* (quantifier symbol)* proposition rightparen* newline | keyword* (quantifier symbol)* leftparen* (quantifier symbol)* proposition rightparen* newline
    proposition: atomicproposition | complexproposition
    complexproposition: keyword* proposition keyword leftparen* (quantifier symbol)* proposition rightparen*
    atomicproposition: leftparen* term* leftparen* term* rightparen*
    !term: (LETTER+) (LETTER+|DIGIT+|"=" | "+" | "-" | "," | "≠")* | (DIGIT+) (LETTER+|DIGIT+|"=" | "+" | "-" | "," | "≠")*
    !leftparen: "("
    !rightparen: ")"
    !keyword: "and" | "not" | "implies" | "or" | "xor" | "implies" | "implies"
    !quantifier: "exists" | "forall"
    symbol: LETTER
    newline: /\n/

    %import common.LETTER
    %import common.DIGIT
    %import common.INT -> NUMBER
    %import common.ESCAPED_STRING -> STRING
    %import common.WS
    %ignore WS
"""
  # prepare prompt
  rag_prompt = f"""
  <start_of_turn>user
  You are an expert logician. You are given a syllogism in natural language with premises between <PREMISES></PREMISES> and conclusion between <CONCLUSION></CONCLUSION> tags and CLIF with premises between <PREMISES-CLIF></PREMISES-CLIF> and conclusion between <CONCLUSION-CLIF></CONCLUSION-CLIF> tags.
  The CLIF BNF grammar to understand and reason in the language is given in the <GRAMMAR></GRAMMAR> tags.
  <GRAMMAR>{grammar}</GRAMMAR>
  <PREMISES>{subject}</PREMISES>
  <CONCLUSION>{obj}</CONCLUSION>
  <PREMISES-CLIF>{subject_clif}</PREMISES-CLIF>
  <CONCLUSION-CLIF>{obj_clif}</CONCLUSION-CLIF>
  Classify the conclusion as "T" if true, "F" if false or "U" if uncertain based on the premises. Present your answer only between <output></output> tags.
  <end_of_turn>
  <start_of_turn>model
  """
  input_ids = tokenizer(rag_prompt, return_tensors="pt").to(model.device)
  response = model.generate(**input_ids, max_new_tokens=500)
  predicted_relation = tokenizer.decode(response[0])
  matches = re.findall('<output>(.*)</output>', predicted_relation, flags=re.DOTALL)
  res = re.findall(r"<output>(.*)", matches[-1])  # from ['</output> tags.\n  <end_of_turn>\n  <start_of_turn>model\n  <output>T'] to ['T']
  predicted_label = res[0] if res else "None" # take first element from list ['T'] to get 'T'

  print("*** Premises: \n", subject)
  print("*** Conclusion: \n", obj)
  print("*** Premises-CLIF: \n", subject_clif)
  print("*** Conclusion-CLIF: \n", obj_clif)
  print("*** True Label: \n", ref_relation)
  print("*** Predicted Label: \n", predicted_label)
  return predicted_label

In [5]:
def infer_from_ontology(dataset, model, tokenizer, mode='default', notation='NL'):
  evaluation_metrics_df = pd.DataFrame(columns=["Accuracy", "Precision", "Recall", "F1"])
  reference_labels = []
  predicted_labels = []
  for index, row in dataset.iterrows():
      conclusion = row["Conclusions - " + notation]
      premises = row["Premises - " + notation]
      conclusion_clif = row["Conclusions - " + "CLIF"]
      premises_clif = row["Premises - " + "CLIF"]
      label = row["Truth Values"]
      if mode.lower() == "grammar":
        # conduct query with RAG retrival of sources
        # set number of candidate answers to consider as half the total triple store axioms
        source_information = """BNF GRAMMAR"""
        print("*** RAG INFORMATION:", source_information)
      # predict answer with model
      predicted_label = predict_answer(model, tokenizer, conclusion, premises, conclusion_clif, premises_clif, label)
      reference_labels.append(label)
      predicted_labels.append(predicted_label)
  # fill evaluation metrics dataframe
  accuracy_metric = accuracy_score(reference_labels, predicted_labels)
  precision_metric = precision_score(reference_labels, predicted_labels, average="macro")
  recall_metric = recall_score(reference_labels, predicted_labels, average="macro")
  f1_metric = f1_score(reference_labels, predicted_labels, average="macro")
  evaluation_metrics_df["Accuracy"] = [accuracy_metric]
  evaluation_metrics_df["Precision"] = [precision_metric]
  evaluation_metrics_df["Recall"] = [recall_metric]
  evaluation_metrics_df["F1"] = [f1_metric]
  print("Classification Report:", classification_report(reference_labels, predicted_labels))
  print("*************** INFERENCE COMPLETE ***************")
  return reference_labels, predicted_labels, evaluation_metrics_df, accuracy_metric, precision_metric, recall_metric, f1_metric

In [6]:
import torch
import json
from tqdm import tqdm
import torch.nn as nn
from torch.optim import Adam
import nltk
import spacy
import string
import evaluate  # Bleu
from torch.utils.data import Dataset, DataLoader, RandomSampler
import pandas as pd
import numpy as np
import transformers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

import warnings
warnings.filterwarnings("ignore")

In [7]:
# login to hugging face to have access to the model
!pip install huggingface_hub
from huggingface_hub import notebook_login
notebook_login()

In [9]:
# try rag search with gemma
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")
# CPU Enabled uncomment below 👇🏽
#model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it")
# GPU Enabled use below 👇🏽
model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it", device_map="auto")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [10]:
# experiment: ZS prediction without Grammar
ref_labels, pred_labels, eval_metrics_df, acc_metric, pr_metric, re_metric, f_metric = infer_from_ontology(pfolio_df, model, tokenizer, mode='default', notation='CLIF')

Streaming output truncated to the last 5000 lines.
named(amyodell, nameodell) and notableperson(amyodell) and named(jackodell, nameodell) and notableperson(jackodell) and named(matsodell, nameodell) and notableperson(matsodell)
british(amyodell) and singer(amyodell) and songwriter(amyodell)
english(jackodell) and toyinventor(jackodell)
*** Conclusion-CLIF: 
 notableperson(jackodell)
*** True Label: 
 T
*** Predicted Label: 
 T
*** Premises: 
 surname(nameodell) and from(nameodell, odellbedfordshire)
mistakenspellingof(nameo'dell, nameodell) and (exists xexists y(family(x) and named(x, nameo'dell) and (not (x=y)) and family(y) and named(y, nameo'dell))
named(amyodell, nameodell) and notableperson(amyodell) and named(jackodell, nameodell) and notableperson(jackodell) and named(matsodell, nameodell) and notableperson(matsodell)
british(amyodell) and singer(amyodell) and songwriter(amyodell)
english(jackodell) and toyinventor(jackodell)
*** Conclusion: 
 named(amyodell, nameodell)
*** Prem

In [11]:
# output results
print("***** ACCURACY *****")
print(acc_metric)
print("***** PRECISION *****")
print(pr_metric)
print("***** RECALL *****")
print(re_metric)
print("***** F1 *****")
print(f_metric)
eval_metrics_df

***** ACCURACY *****
0.5016611295681063
***** PRECISION *****
0.3993179880647911
***** RECALL *****
0.4839233388770096
***** F1 *****
0.4039980009995003


,Accuracy,Precision,Recall,F1
0,0.501661,0.399318,0.483923,0.403998


In [ ]:
# empty torch cuda cache
torch.cuda.empty_cache()

# delete model from cpu
del(model)